# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/whozahm3d/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Lane:** Freestyle — AI Referral Opportunity Scoring, extended to the full warehouse.

**Question:** Among pages that already have strong organic search visibility, which ones look like plausible candidates for AI-referral pickup based on traits shared with pages that already receive AI-referral sessions — and can those candidates be ranked in a way that beats a naive impressions-only rule?

**Why this, at capstone scale:** Weeks 1–4 established this on the 30,000-row starter sample: AI-referral sessions are rare (6.43% of pages), concentrated among higher-impression, longer-content pages, and — counterintuitively — not concentrated among top-position pages. The capstone moves the same question onto the full `FlyRank/internship-warehouse` release (~79M rows, `fact_content_daily_performance` + `fact_content_query_90d`), where the AI-referral signal (`sessions_ai`) can be aggregated over a properly windowed, leakage-safe period instead of a single fixed 90-day snapshot, and where the client-concentration issue found in Week 2 can be tested at real scale rather than assumed away.

**The decision this supports:** which pages a content team should prioritize when trying to grow AI-tool referral traffic — a prioritization problem with no existing tooling at FlyRank today.

**The action a strategist could take:** given a ranked, reason-coded list of AI-opportunity pages (high search visibility, structurally similar to pages that already get AI sessions, but currently receiving none), restructure those pages first — clearer definitions, more explicit answer framing — and monitor for AI-session lift afterward.

**The cost of a wrong call:** low but not zero. A false positive costs editorial time on a page that doesn't lift. The larger risk, as in Week 1, is overclaiming: this model ranks *opportunity*, not *causation* — it cannot show that any content trait causes AI pickup, only that AI-session pages and candidate pages share observable traits.

**What's new versus Weeks 1–4:** full warehouse scale instead of the 30k sample; a defined, leakage-checked aggregation window for `sessions_ai` instead of a static column; validation against a real train/test split rather than descriptive stats alone; and — if this holds up — a stretch attempt at whether AI-opportunity pages that get restructured show measurable session growth in a later window (tying into the growth/momentum stretch goal noted in Week 1).

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/whozahm3d/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

In [27]:
# Anchor check: re-confirm the starter-sample numbers this question is built on
# (full warehouse-scale version of this comes in Section 2 — this is just grounding, not new analysis)
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
has_ai = df["ai_sessions_90d"] > 0

print(f"AI-session pages: {has_ai.mean()*100:.2f}% ({has_ai.sum()} of {len(df)})")
print(f"Median impressions_90d — no AI: {df[~has_ai]['impressions_90d'].median():.0f} | has AI: {df[has_ai]['impressions_90d'].median():.0f}")
print(f"Median avg_position   — no AI: {df[~has_ai]['avg_position'].median():.1f} | has AI: {df[has_ai]['avg_position'].median():.1f}")

AI-session pages: 6.43% (1930 of 30000)
Median impressions_90d — no AI: 630 | has AI: 8014
Median avg_position   — no AI: 10.5 | has AI: 15.3


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** `FlyRank/internship-warehouse` (Hugging Face, gated), frozen snapshot export 2026-07-03.

**Tables used:**
- `fact_content_daily_performance` — grain `(client_hash_id, content_hash_id, report_date)`, partitioned by `month=YYYY-MM`. This is the daily source for both the label (`sessions_ai`) and the rolled-up features (`gsc_impressions`, `gsc_clicks`, `gsc_avg_position`).
- `dim_content` — static content attributes, joined for `word_count`.
- `dim_clients` — joined only to interpret `access_profile` against row-level flags, never used to filter directly (per the stale-label finding below).
- `fact_content_query_90d` — not used in this section; reserved for a possible query-level feature pass in Methodology if the article-level score needs more signal.

**Date window:** the data contract (ML-04) established `month=2026-03` as the safe development month, with the `_sample` file (June 2026) sealed off as the natural outcome window for any past→future label — reusing it here for development would leak the future into training. The capstone keeps that boundary: development and feature-building stay in past months; if a forward-looking label is attempted (Methodology, §3), the held-out future window is defined explicitly there, not borrowed from `_sample`.

Because ML-04 showed the AI-referral base rate depends heavily on the aggregation window (0.0562% daily → 1.15% monthly → 6.43% at 90 days in the starter CSV), this capstone aggregates `sessions_ai` over a wider multi-month window rather than a single calendar month, to get a base rate closer to what the scoring work in ML-03 was actually tuned against. The exact window is fixed and verified below.

**Exclusions carried forward from the ML-04 contract, with reasons:**
- Rows on `dim_content.is_deleted = True` — a deleted article's past performance says nothing about future opportunity (~2% of fact rows in the March slice).
- Articles with fewer than 14 days of history in the window — insufficient for a stable rolled-up signal; the same 14-day floor used for label-window computability in ML-04/ML-05.
- `client_hash_id` filtering by `dim_clients.access_profile` is **not** used — ML-04 found this field stale for at least 2 clients (`no_search_or_analytics_access` clients with real GSC data present). Row-level `gsc_data_available` / `ga4_data_available` flags drive filtering instead.
- Rows where `ga4_data_available IS NULL` (true "no GA4 access," ~31% of rows) are excluded from any `sessions_ai`-based calculation — this is structurally missing, not a measured zero. Rows where the flag is `False` (measured zero engagement) are kept and treated as real zeros, not dropped.

**Public-safe note:** no client names, domains, or raw query strings are retained past this section — all identifiers used downstream are the salted hash keys already provided by the release.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
from huggingface_hub import login, hf_hub_download
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))
repo = "FlyRank/internship-warehouse"

# TODO: confirm final window (list of month partitions) before running at scale
MONTHS = ["2026-03"]  # placeholder — extend once window is finalized

fact_paths = [
    hf_hub_download(repo_id=repo, repo_type="dataset",
                     filename=f"fact_content_daily_performance/month={m}/data_0.parquet")
    for m in MONTHS
]

dim_clients_path = hf_hub_download(repo_id=repo, repo_type="dataset", filename="dim_clients.parquet")
dim_content_path = hf_hub_download(repo_id=repo, repo_type="dataset", filename="dim_content.parquet")

con = duckdb.connect()
con.sql(f"CREATE OR REPLACE VIEW fact AS SELECT * FROM read_parquet({fact_paths})")
con.sql(f"CREATE OR REPLACE VIEW dim_clients AS SELECT * FROM read_parquet('{dim_clients_path}')")
con.sql(f"CREATE OR REPLACE VIEW dim_content AS SELECT * FROM read_parquet('{dim_content_path}')")

for view in ["fact", "dim_clients", "dim_content"]:
    n = con.sql(f"SELECT COUNT(*) AS n_rows FROM {view}").df()["n_rows"][0]
    print(f"{view}: {n:,} rows")

fact: 9,841,378 rows
dim_clients: 104 rows
dim_content: 519,606 rows


In [4]:
# Exclusion counts — how much of the raw window survives each filter
con.sql("""
    WITH base AS (
        SELECT f.*, d.is_deleted, d.word_count
        FROM fact f
        LEFT JOIN dim_content d USING (content_hash_id)
    ),
    history AS (
        SELECT content_hash_id, COUNT(DISTINCT report_date) AS days_present
        FROM fact
        GROUP BY content_hash_id
    )
    SELECT
        COUNT(*) AS raw_rows,
        SUM(CASE WHEN is_deleted THEN 1 ELSE 0 END) AS deleted_content_rows,
        SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END) AS no_ga4_access_rows
    FROM base
""").df()

,raw_rows,deleted_content_rows,no_ga4_access_rows
0,9841378,201090.0,3018741.0


In [5]:
# Article-level history depth check against the 14-day floor
history = con.sql("""
    SELECT content_hash_id, COUNT(DISTINCT report_date) AS days_present
    FROM fact
    GROUP BY content_hash_id
""").df()

print(f"Articles total: {len(history):,}")
print(f"Articles below 14-day floor: {(history['days_present'] < 14).sum():,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Articles total: 331,437
Articles below 14-day floor: 10,755


In [6]:
# Grain check — should return 0 rows
grain_check = con.sql("""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM fact
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
""").df()
print("Duplicate grain rows found:", len(grain_check))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found: 0


In [7]:
# Base rate for the chosen window, after applying the stated exclusions
# (deleted content, GA4-access rows only, 14-day-history articles only)
eligible = con.sql("""
    WITH history AS (
        SELECT content_hash_id, COUNT(DISTINCT report_date) AS days_present
        FROM fact
        GROUP BY content_hash_id
    ),
    filtered AS (
        SELECT f.content_hash_id, f.sessions_ai
        FROM fact f
        JOIN dim_content d USING (content_hash_id)
        JOIN history h USING (content_hash_id)
        WHERE d.is_deleted = FALSE
          AND f.ga4_data_available IS NOT NULL
          AND h.days_present >= 14
    )
    SELECT
        content_hash_id,
        SUM(sessions_ai) AS sessions_ai_window
    FROM filtered
    GROUP BY content_hash_id
""").df()

has_ai = eligible["sessions_ai_window"] > 0
print(f"Eligible articles: {len(eligible):,}")
print(f"Base rate this window: {has_ai.mean()*100:.2f}% ({has_ai.sum():,} of {len(eligible):,})")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible articles: 245,550
Base rate this window: 1.54% (3,773 of 245,550)


In [8]:
# word_count coverage on the eligible article set
con.sql("""
    SELECT
        COUNT(*) AS n_content_rows,
        COUNT(word_count) AS n_with_word_count,
        ROUND(100.0 * COUNT(word_count) / COUNT(*), 2) AS pct_with_word_count
    FROM dim_content
    WHERE content_hash_id IN (SELECT DISTINCT content_hash_id FROM fact)
""").df()

,n_content_rows,n_with_word_count,pct_with_word_count
0,331437,224008,67.59


**Verified on `month=2026-03`:**

- *Grain confirmed: 0 duplicate `(client_hash_id, content_hash_id, report_date)` rows.*
- *Exclusions applied: 201,090 rows on deleted content (~2.0%), 3,018,741 rows with no GA4 access (~30.7%), 10,755 of 331,437 articles below the 14-day history floor.*
- *After filtering, 245,550 eligible articles remain. Base rate: **1.54%** (3,773 articles) show any `sessions_ai` activity — higher than ML-04's unfiltered 1.15% (the filters removed noise, not signal — expected direction), but still well below the starter CSV's 90-day 6.43%.*
- *`word_count` coverage on the eligible-article set: 67.59% — matches ML-04's finding exactly, confirming the gap isn't month-specific.*

***Note before moving on:** `MONTHS` is still just `["2026-03"]` — the placeholder was never extended. A single calendar month is why the base rate (1.54%) still sits well under the 90-day 6.43% reference; ML-04 already showed this gap shrinks as the window widens. Before locking Section 2, decide whether to pull additional month partitions to approximate a 90-day window, or explicitly keep single-month scope and note the base-rate gap as a stated limitation rather than closing it.*

In [9]:
# Extending window toward ~90 days, staying strictly before the sealed June _sample
MONTHS = ["2026-01", "2026-02", "2026-03"]

fact_paths = [
    hf_hub_download(repo_id=repo, repo_type="dataset",
                     filename=f"fact_content_daily_performance/month={m}/data_0.parquet")
    for m in MONTHS
]

dim_clients_path = hf_hub_download(repo_id=repo, repo_type="dataset", filename="dim_clients.parquet")
dim_content_path = hf_hub_download(repo_id=repo, repo_type="dataset", filename="dim_content.parquet")

con = duckdb.connect()
con.sql(f"CREATE OR REPLACE VIEW fact AS SELECT * FROM read_parquet({fact_paths})")
con.sql(f"CREATE OR REPLACE VIEW dim_clients AS SELECT * FROM read_parquet('{dim_clients_path}')")
con.sql(f"CREATE OR REPLACE VIEW dim_content AS SELECT * FROM read_parquet('{dim_content_path}')")

for view in ["fact", "dim_clients", "dim_content"]:
    n = con.sql(f"SELECT COUNT(*) AS n_rows FROM {view}").df()["n_rows"][0]
    print(f"{view}: {n:,} rows")

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 90.3MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 89.0MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact: 25,087,303 rows
dim_clients: 104 rows
dim_content: 519,606 rows


In [10]:
# Exclusion counts — how much of the raw window survives each filter
con.sql("""
    WITH base AS (
        SELECT f.*, d.is_deleted, d.word_count
        FROM fact f
        LEFT JOIN dim_content d USING (content_hash_id)
    ),
    history AS (
        SELECT content_hash_id, COUNT(DISTINCT report_date) AS days_present
        FROM fact
        GROUP BY content_hash_id
    )
    SELECT
        COUNT(*) AS raw_rows,
        SUM(CASE WHEN is_deleted THEN 1 ELSE 0 END) AS deleted_content_rows,
        SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END) AS no_ga4_access_rows
    FROM base
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,raw_rows,deleted_content_rows,no_ga4_access_rows
0,25087303,1169317.0,12615776.0


In [11]:
# Article-level history depth check against the 14-day floor
history = con.sql("""
    SELECT content_hash_id, COUNT(DISTINCT report_date) AS days_present
    FROM fact
    GROUP BY content_hash_id
""").df()

print(f"Articles total: {len(history):,}")
print(f"Articles below 14-day floor: {(history['days_present'] < 14).sum():,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Articles total: 349,411
Articles below 14-day floor: 10,761


In [12]:
# Grain check — should return 0 rows
grain_check = con.sql("""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM fact
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
""").df()
print("Duplicate grain rows found:", len(grain_check))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found: 0


In [13]:
# Base rate for the chosen window, after applying the stated exclusions
# (deleted content, GA4-access rows only, 14-day-history articles only)
eligible = con.sql("""
    WITH history AS (
        SELECT content_hash_id, COUNT(DISTINCT report_date) AS days_present
        FROM fact
        GROUP BY content_hash_id
    ),
    filtered AS (
        SELECT f.content_hash_id, f.sessions_ai
        FROM fact f
        JOIN dim_content d USING (content_hash_id)
        JOIN history h USING (content_hash_id)
        WHERE d.is_deleted = FALSE
          AND f.ga4_data_available IS NOT NULL
          AND h.days_present >= 14
    )
    SELECT
        content_hash_id,
        SUM(sessions_ai) AS sessions_ai_window
    FROM filtered
    GROUP BY content_hash_id
""").df()

has_ai = eligible["sessions_ai_window"] > 0
print(f"Eligible articles: {len(eligible):,}")
print(f"Base rate this window: {has_ai.mean()*100:.2f}% ({has_ai.sum():,} of {len(eligible):,})")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible articles: 266,175
Base rate this window: 2.20% (5,851 of 266,175)


In [14]:
# word_count coverage on the eligible article set
con.sql("""
    SELECT
        COUNT(*) AS n_content_rows,
        COUNT(word_count) AS n_with_word_count,
        ROUND(100.0 * COUNT(word_count) / COUNT(*), 2) AS pct_with_word_count
    FROM dim_content
    WHERE content_hash_id IN (SELECT DISTINCT content_hash_id FROM fact)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_content_rows,n_with_word_count,pct_with_word_count
0,349411,241982,69.25


**Verified on the extended window (`2026-01`, `2026-02`, `2026-03` — three months, still strictly before the sealed June `_sample`):**

- *Grain confirmed: 0 duplicate `(client_hash_id, content_hash_id, report_date)` rows across 25,087,303 raw rows.*
- *Exclusions applied: 1,169,317 rows on deleted content (4.66%), 12,615,776 rows with no GA4 access (50.28% — notably higher than March alone's 30.7%, likely reflecting clients whose `ga4_data_start` falls later in the window and so have no GA4 access in Jan/Feb at all).*
- *Article history: 10,761 of 349,411 articles fall below the 14-day floor — essentially unchanged from the March-only count (10,755), confirming this floor is governed by individual article age, not window length.*
- *266,175 eligible articles remain after filtering. Base rate: **2.20%** (5,851 articles) show any `sessions_ai` activity over the 3-month window — up from March-only's 1.54%, moving in the expected direction as window length increases (per ML-04's daily → monthly → 90-day progression: 0.06% → 1.15–1.54% → 6.43%).*
- *`word_count` coverage: 69.25% — stable with the March-only figure (67.59%), confirming this is a structural gap in `dim_content`, not a month-specific artifact.*

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Assumptions carried forward:**
- This stays a ranking/scoring task, never a binary classifier — confirmed again at capstone scale: even at a 3-month window, the AI-referral base rate is 2.20% (5,851 of 266,175 eligible articles). Too sparse and too imbalanced for honest classification metrics.
- The unit of analysis is one article, scored once over the fixed 3-month window (2026-01 to 2026-03) defined and filtered in Section 2.
- Client concentration (found in ML-02, on the starter CSV) is treated as a real risk at warehouse scale, not just a starter-data quirk — addressed directly in the validation split below, not just noted as a limitation.

**Features (all knowable independent of the label):**
- `word_count` — from `dim_content`, static content attribute (69.25% coverage in this window; `has_word_count` flag used rather than `fillna(0)`, per the ML-01 finding that missingness correlates with content type).
- `gsc_impressions_sum` — total search impressions across the window, rolled up from daily `gsc_impressions`. Same role as `impressions_90d` in ML-02/ML-03, now computed directly from daily rows instead of a pre-aggregated column.
- `gsc_avg_position_mean` — mean of daily `gsc_avg_position` across days where `gsc_data_available IS TRUE`. NULLs are skipped, not zero-filled, per the ML-04 finding that a missing position means "nothing to rank," not "position zero."

None of these three features are derived from `sessions_ai` or `ga4_sessions` in any way — they come entirely from the GSC side of the panel, keeping the label and the features causally separate data sources within the same window.

**Label:** `sessions_ai_window > 0` (binary indicator) — the eligible-article label already built in Section 2, filtered to deleted-content-excluded, GA4-access rows only, ≥14-day history.

**Score:** an extension of ML-03's `ai_opportunity_score` — a z-scored composite of `gsc_impressions_sum` and `word_count`, built on the full eligible-article set rather than the 30,000-row starter sample. `gsc_avg_position_mean` is computed and retained as a diagnostic column (per the counterintuitive ML-01 finding that AI-session pages don't skew toward top positions) but is not weighted into the score itself, to avoid encoding an assumption the data has already contradicted once.

**Baseline:** a naive top-impressions rule — rank purely by `gsc_impressions_sum` — mirroring the 34.00% precision@500 naive baseline from ML-02, now re-measured on this window's eligible set and label.

**Validation design:** grouped split by `client_hash_id`, not a random row split. ML-02 found top-ranked scores concentrated in a single client on the starter data; a random split risks the same client's articles leaking across train and test, inflating apparent lift. Splitting whole clients into train/test groups keeps that risk out of the evaluation honestly.

**Leakage check plan:** following the ML-03 leakage notebook's approach — deliberately inject a feature derived from `sessions_ai` itself (e.g., prior-day AI sessions) into a second version of the score, and confirm it produces an inflated, dishonest lift@K compared to the real score. This becomes a code cell in Section 4 alongside the honest result, not run here.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Build the eligible-article feature table: word_count, rolled-up impressions, mean position, label
features = con.sql("""
    WITH history AS (
        SELECT content_hash_id, COUNT(DISTINCT report_date) AS days_present
        FROM fact
        GROUP BY content_hash_id
    ),
    eligible AS (
        SELECT f.content_hash_id, f.client_hash_id, f.report_date,
               f.gsc_impressions, f.gsc_avg_position, f.gsc_data_available,
               f.sessions_ai
        FROM fact f
        JOIN dim_content d USING (content_hash_id)
        JOIN history h USING (content_hash_id)
        WHERE d.is_deleted = FALSE
          AND f.ga4_data_available IS NOT NULL
          AND h.days_present >= 14
    )
    SELECT
        content_hash_id,
        ANY_VALUE(client_hash_id) AS client_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_sum,
        AVG(CASE WHEN gsc_data_available THEN gsc_avg_position END) AS gsc_avg_position_mean,
        SUM(sessions_ai) AS sessions_ai_window
    FROM eligible
    GROUP BY content_hash_id
""").df()

print(f"Feature table: {len(features):,} articles")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature table: 266,175 articles


,content_hash_id,client_hash_id,gsc_impressions_sum,gsc_avg_position_mean,sessions_ai_window
0,content_65b8a610174a1036,client_62f4a7e64f5e0096,3535.0,5.141441,0.0
1,content_80071808216aef39,client_62f4a7e64f5e0096,77501.0,4.587268,0.0
2,content_9a136f9ba3924c74,client_62f4a7e64f5e0096,35356.0,3.706159,0.0
3,content_200d6a6d1cac69b7,client_62f4a7e64f5e0096,3313.0,0.499461,0.0
4,content_0d440941b1c6807f,client_62f4a7e64f5e0096,4580.0,3.890039,0.0


In [16]:
# Join word_count and build has_word_count flag (no fillna(0) per ML-01 finding)
word_counts = con.sql("SELECT content_hash_id, word_count FROM dim_content").df()
features = features.merge(word_counts, on="content_hash_id", how="left")
features["has_word_count"] = features["word_count"].notna()

features["label_ai"] = (features["sessions_ai_window"] > 0).astype(int)
print(f"Label positive rate: {features['label_ai'].mean()*100:.2f}%")

Label positive rate: 2.20%


In [17]:
import numpy as np

# z-scored composite score (word_count NaNs excluded from the z-score calc, not zero-filled)
def zscore(s):
    return (s - s.mean()) / s.std()

features["z_impressions"] = zscore(features["gsc_impressions_sum"])
features["z_word_count"] = zscore(features["word_count"])  # NaN stays NaN, not silently 0

features["ai_opportunity_score"] = features[["z_impressions", "z_word_count"]].mean(axis=1, skipna=True)

# baseline: naive top-impressions rank
features["baseline_score"] = features["gsc_impressions_sum"]

features[["content_hash_id", "ai_opportunity_score", "baseline_score", "label_ai"]].describe()

,ai_opportunity_score,baseline_score,label_ai
count,266175.0,266175.000000,266175.000000
mean,-0.021772,1223.464623,0.021982
std,0.718544,6062.286807,0.146624
min,-1.0699,0.000000,0.000000
25%,-0.368554,0.000000,0.000000
50%,-0.188289,5.000000,0.000000
75%,0.142852,333.000000,0.000000
max,68.4918,830289.000000,1.000000


In [18]:
# Grouped train/test split by client_hash_id — whole clients assigned, not rows
rng = np.random.default_rng(42)
clients = features["client_hash_id"].unique()
rng.shuffle(clients)

split_idx = int(len(clients) * 0.8)
train_clients = set(clients[:split_idx])
test_clients = set(clients[split_idx:])

features["split"] = np.where(features["client_hash_id"].isin(train_clients), "train", "test")

print(features["split"].value_counts())
print(f"\nTrain clients: {len(train_clients)} | Test clients: {len(test_clients)}")
print(f"Train label rate: {features[features.split=='train']['label_ai'].mean()*100:.2f}%")
print(f"Test label rate: {features[features.split=='test']['label_ai'].mean()*100:.2f}%")

split
train    191216
test      74959
Name: count, dtype: int64

Train clients: 31 | Test clients: 8
Train label rate: 1.15%
Test label rate: 4.87%


In [19]:
# 1. Grain check on the feature table
dupe_check = features.groupby("content_hash_id").size()
print("Articles with duplicate rows in feature table:", (dupe_check > 1).sum())

Articles with duplicate rows in feature table: 0


In [20]:
# 2. Client concentration in top-ranked scores (the ML-02 concern, tested at capstone scale)
for split_name in ["train", "test"]:
    split_df = features[features.split == split_name]
    top_k = split_df.sort_values("ai_opportunity_score", ascending=False).head(500)
    top_client_share = top_k["client_hash_id"].value_counts(normalize=True).iloc[0]
    n_unique_clients = top_k["client_hash_id"].nunique()
    print(f"{split_name}: top single client holds {top_client_share*100:.1f}% of top-500; "
          f"{n_unique_clients} unique clients represented")

train: top single client holds 42.0% of top-500; 9 unique clients represented
test: top single client holds 95.6% of top-500; 5 unique clients represented


In [21]:
# 3. Article count balance per split (not just client count)
print(features["split"].value_counts())
print("\nShare of articles per split:")
print((features["split"].value_counts(normalize=True) * 100).round(1))

split
train    191216
test      74959
Name: count, dtype: int64

Share of articles per split:
split
train    71.8
test     28.2
Name: proportion, dtype: float64


In [22]:
# 4. Does word_count add signal beyond impressions, or is it mostly noise?
corr = features[["z_impressions", "z_word_count"]].corr().iloc[0, 1]
print(f"Correlation between z_impressions and z_word_count: {corr:.3f}")

# How often does adding word_count change the top-500 ranking vs. impressions alone?
top_impr_only = set(features.sort_values("gsc_impressions_sum", ascending=False).head(500)["content_hash_id"])
top_composite = set(features.sort_values("ai_opportunity_score", ascending=False).head(500)["content_hash_id"])
overlap = len(top_impr_only & top_composite)
print(f"Overlap between impressions-only top-500 and composite-score top-500: {overlap}/500")

Correlation between z_impressions and z_word_count: 0.127
Overlap between impressions-only top-500 and composite-score top-500: 417/500


In [23]:
# Check per-client label rate and size — needed to build a smarter split
client_stats = features.groupby("client_hash_id").agg(
    n_articles=("content_hash_id", "count"),
    label_rate=("label_ai", "mean")
).reset_index().sort_values("n_articles", ascending=False)

client_stats

,client_hash_id,n_articles,label_rate
16,client_625b6439094e23e4,31887,0.000157
12,client_3ffa76342f366962,31077,0.001319
19,client_73cda7b4e4f265ea,28225,0.004606
17,client_62f4a7e64f5e0096,20623,0.000000
8,client_23a62021009f63c4,14016,0.215539
18,client_65de48885f4ef01b,13333,0.011025
28,client_ba65e80a1116ae41,13129,0.020565
37,client_fef1a8f436438636,10960,0.015055
10,client_3197e6291363b4db,10168,0.013080
34,client_e547b89c05043229,9300,0.047097


In [24]:
# Within-client normalized score (ML-02 mitigation for client concentration)
# z-score impressions and word_count WITHIN each client, not across the whole set —
# so a client's own top articles rank highly regardless of that client's overall size/volume

def zscore_within_client(df, col):
    grouped = df.groupby("client_hash_id")[col]
    mean = grouped.transform("mean")
    std = grouped.transform("std")
    return (df[col] - mean) / std

features["z_impressions_within"] = zscore_within_client(features, "gsc_impressions_sum")
features["z_word_count_within"] = zscore_within_client(features, "word_count")

features["ai_opportunity_score_within_client"] = features[
    ["z_impressions_within", "z_word_count_within"]
].mean(axis=1, skipna=True)

features[["content_hash_id", "client_hash_id", "ai_opportunity_score",
          "ai_opportunity_score_within_client"]].head()

,content_hash_id,client_hash_id,ai_opportunity_score,ai_opportunity_score_within_client
0,content_65b8a610174a1036,client_62f4a7e64f5e0096,0.381298,0.048842
1,content_80071808216aef39,client_62f4a7e64f5e0096,6.315726,4.251767
2,content_9a136f9ba3924c74,client_62f4a7e64f5e0096,2.998162,2.125228
3,content_200d6a6d1cac69b7,client_62f4a7e64f5e0096,0.344678,0.022103
4,content_0d440941b1c6807f,client_62f4a7e64f5e0096,0.505958,0.391556


In [25]:
# Re-check concentration using the within-client score instead of the raw composite
for split_name in ["train", "test"]:
    split_df = features[features.split == split_name]
    top_k = split_df.sort_values("ai_opportunity_score_within_client", ascending=False).head(500)
    top_client_share = top_k["client_hash_id"].value_counts(normalize=True).iloc[0]
    n_unique_clients = top_k["client_hash_id"].nunique()
    print(f"{split_name} (within-client score): top single client holds "
          f"{top_client_share*100:.1f}% of top-500; {n_unique_clients} unique clients represented")

train (within-client score): top single client holds 22.6% of top-500; 29 unique clients represented
test (within-client score): top single client holds 38.0% of top-500; 8 unique clients represented


In [26]:
# Client-level diagnostics — needed before fixing the split itself
client_stats = features.groupby("client_hash_id").agg(
    n_articles=("content_hash_id", "count"),
    label_rate=("label_ai", "mean")
).reset_index().sort_values("n_articles", ascending=False)

client_stats

,client_hash_id,n_articles,label_rate
16,client_625b6439094e23e4,31887,0.000157
12,client_3ffa76342f366962,31077,0.001319
19,client_73cda7b4e4f265ea,28225,0.004606
17,client_62f4a7e64f5e0096,20623,0.000000
8,client_23a62021009f63c4,14016,0.215539
18,client_65de48885f4ef01b,13333,0.011025
28,client_ba65e80a1116ae41,13129,0.020565
37,client_fef1a8f436438636,10960,0.015055
10,client_3197e6291363b4db,10168,0.013080
34,client_e547b89c05043229,9300,0.047097


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
